# ML-03 — Frame Your Lane as an ML Task

**Chosen lane: CTR / Engagement Opportunity Scoring. Initial scope: CTR.**

This continues my ML-02 research question: which visible pages under-capture clicks compared with position/content-type peers and deserve editorial review first? Engagement remains possible later context; it is not a second target in this notebook.

Prepared with AI assistance. The code calculates the evidence from the bundled starter CSV; the framing below explains the decision and the limits. Run all cells with Python, pandas, and numpy. The loader supports the repository root, starter root, or notebook folder. References: `docs/data-dictionary.md`, `docs/ml-intern-dataset-and-lane-guide.md`, `skills/framing-ml-problems/SKILL.md`, and `skills/flyrank/flyrank-data/SKILL.md` inside the starter folder.

## 1. My lane as an ML task (type)

**Task type: ranking / scoring for decision support.** The output orders pages for a content editor with a provisional budget of 20 reviews per cycle. A binary low-CTR flag cannot tell the editor which 20 to inspect first, and clustering would answer a different question.

For a content editor deciding which pages to inspect, I will build a position-adjusted CTR review queue from observed search counts and content context. The initial score is a transparent peer-gap proxy, evaluated eventually with independent editorial **precision@20**. A wrong recommendation wastes review time and an unnecessary edit may damage a useful page; a missed recommendation leaves an opportunity unreviewed. Data may help because position, exposure, content type, and client/query mix interact. I claim only measured, observational decision support, with no learned-model or causal claim.

**Actor and action:** the editor checks intent, title/meta, and search-result context in an authorized workflow, then chooses an appropriate snippet/metadata or intent-match edit, or monitoring. Low CTR alone does not establish the cause or justify an automatic rewrite.

The initial review policy is at least **500 impressions over 90 days** and recorded average position **1–20**, continuing ML-02. Position 0 means missing; positive values below 1 need an audit. These thresholds and K=20 are policy assumptions, not universal SEO rules.

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

relative_csv = Path("data/raw/content_refresh_anonymized.csv")
root = next(
    (candidate for base in (Path.cwd(), *Path.cwd().parents)
     for candidate in (base, base / "flyrank-ml-internship-starter-main")
     if (candidate / relative_csv).is_file()), None
)
if root is None:
    raise FileNotFoundError("Run inside a clone containing the starter CSV.")
df = pd.read_csv(root / relative_csv)
required = {"content_id", "client_id", "content_type", "impressions_90d",
            "clicks_90d", "avg_position", "ctr"}
assert required.issubset(df.columns)
assert df["content_id"].notna().all() and df["content_id"].is_unique
assert df["client_id"].notna().all()
assert df[["clicks_90d", "impressions_90d"]].notna().all().all()
assert df[["clicks_90d", "impressions_90d"]].ge(0).all().all()
assert df["clicks_90d"].le(df["impressions_90d"]).all()

lane = df.loc[df["impressions_90d"].ge(500)
              & df["avg_position"].between(1, 20)].copy()
assert not lane.empty
assert lane[["content_type", "ctr"]].notna().all().all()
# Recompute exact CTR from counts. Stored rate columns are percentages.
lane["observed_ctr_pct"] = 100 * lane["clicks_90d"] / lane["impressions_90d"]
assert np.allclose(lane["ctr"], lane["observed_ctr_pct"], atol=0.0051, rtol=0)
lane["position_band"] = pd.cut(
    lane["avg_position"], [0, 3, 10, 20], labels=["1–3", ">3–10", ">10–20"]
)
print(f"Starter: {len(df):,} pages × {df.shape[1]} columns; {df['client_id'].nunique()} clients.")
print(f"CTR lane: {len(lane):,} eligible pages; {lane['client_id'].nunique()} clients.")
print(f"Missing-position rows excluded: {df['avg_position'].eq(0).sum():,}.")

Starter: 30,000 pages × 44 columns; 32 clients.
CTR lane: 12,009 eligible pages; 28 clients.
Missing-position rows excluded: 1,205.


## 2. Target or proxy

The underlying observations are **clicks and impressions**. My current continuous proxy is the positive CTR gap relative to peers, weighted by exposure:

`ctr_gap_pp = max(peer_ctr_pct − observed_ctr_pct, 0)`

`opportunity_proxy = impressions_90d × ctr_gap_pp / 100`

The score has click-count units, but is **not a prediction of recoverable clicks**. It is a rule-defined review heuristic derived from measured counts, not an observed label proving that an edit will work. I will not train a model to reproduce it and call that discovery.

Peers share position band and content type. Their benchmark is pooled clicks / pooled impressions, not the mean of rounded page rates. I exclude the candidate's entire client from its benchmark so its own clicks cannot pull the benchmark toward itself. I require at least 30 peer pages from at least 3 other clients; unsupported comparisons remain missing. These support thresholds are provisional. Excluding clients limits self-influence but does not establish future validity or remove client-mix confounding.

**Independent evaluation target:** an editor's recorded `review_worth_action` judgment (1/0 under a rubric). It is currently missing, not automatically inferred from the score. A future observed target could be next-30-day CTR from daily warehouse facts, with earlier-only features and time/client validation. The 90-day position/context here overlaps the comparison windows, so I do not use it to claim future prediction. Showing recent and previous 30-day columns alone would not fix that overlap.

In [2]:
# Leave-one-client-out pooled peer benchmark; IDs only group the data.
peer_keys = ["position_band", "content_type"]
grouped = lane.groupby(peer_keys, observed=True)
client_grouped = lane.groupby(peer_keys + ["client_id"], observed=True)
other_impressions = (grouped["impressions_90d"].transform("sum")
                     - client_grouped["impressions_90d"].transform("sum"))
other_clicks = (grouped["clicks_90d"].transform("sum")
                - client_grouped["clicks_90d"].transform("sum"))
lane["peer_pages_other_clients"] = (grouped["content_id"].transform("size")
                                     - client_grouped["content_id"].transform("size"))
lane["peer_clients_other"] = grouped["client_id"].transform("nunique") - 1
supported = (lane["peer_pages_other_clients"].ge(30)
             & lane["peer_clients_other"].ge(3) & other_impressions.gt(0))
lane["peer_ctr_pct"] = (100 * other_clicks / other_impressions).where(supported)
lane["ctr_gap_pp"] = (lane["peer_ctr_pct"] - lane["observed_ctr_pct"]).clip(lower=0)
lane["opportunity_proxy"] = lane["impressions_90d"] * lane["ctr_gap_pp"] / 100
lane["review_worth_action"] = pd.Series(pd.NA, index=lane.index, dtype="Int64")

assert lane.loc[~supported, "opportunity_proxy"].isna().all()
assert lane.loc[supported, "opportunity_proxy"].ge(0).all()
assert lane["review_worth_action"].isna().all()
print(f"Supported peer comparisons: {supported.sum():,}/{len(lane):,}.")
print(f"Pages with a positive proxy gap: {lane['opportunity_proxy'].gt(0).sum():,}.")
# Five real pages, omitting grouping IDs from the public display.
display(lane[["observed_ctr_pct", "peer_ctr_pct", "ctr_gap_pp",
              "opportunity_proxy", "review_worth_action"]].head().round(4))

Supported peer comparisons: 11,911/12,009.
Pages with a positive proxy gap: 8,429.


,observed_ctr_pct,peer_ctr_pct,ctr_gap_pp,opportunity_proxy,review_worth_action
0,0.7626,0.3545,0.0000,0.0000,<NA>
3,0.4936,0.4228,0.0000,0.0000,<NA>
5,0.0252,0.3541,0.3289,13.0563,<NA>
9,0.1613,0.4228,0.2615,3.2422,<NA>
10,1.5488,0.3514,0.0000,0.0000,<NA>


## 3. Success metric

**Primary metric: independent editorial precision@20** = number of actionable pages among the top 20 / 20. The reviewer checks that sufficient volume exists and that an intent/snippet inspection identifies a concrete, justified action; low CTR by itself is not an actionable judgment. Record the review before any outcome claim, ideally without showing the reviewer which ranking selected the page.

Compare the proxy queue with an **impressions-only baseline** at the same K and eligibility policy. A provisional success bar is at least **12 actionable reviews out of 20 (0.60)** and at least **2 more actionable pages than the baseline (+0.10)**, then confirm this with the editor and repeat across clients/cycles. These are chosen goals, not measured results. They measure review usefulness, not causal traffic uplift.

The starter has no independent editorial labels, so I cannot honestly compute precision@20 today. The code explicitly returns missing until all 20 labels exist. It computes **queue coverage** (an operational diagnostic) on both baselines now. Creating a `low_ctr` label from my own score and evaluating against it would be circular. Later validation needs independent reviews; future-CTR modeling needs separate windows and a held-out-client/time evaluation.

In [3]:
K = 20
candidate_pool = lane.loc[lane["opportunity_proxy"].gt(0)].copy()
proxy_queue = candidate_pool.sort_values(
    ["opportunity_proxy", "impressions_90d"], ascending=False, kind="stable"
).head(K)
# Same eligible/supported pool for both queues; baseline does not use CTR gap.
impressions_queue = lane.loc[supported].sort_values(
    "impressions_90d", ascending=False, kind="stable"
).head(K)

def editorial_precision_at_k(queue, k=20):
    labels = queue.head(k)["review_worth_action"]
    if len(labels) < k or labels.isna().any():
        return np.nan  # Missing judgments must not be counted as false negatives.
    assert labels.isin([0, 1]).all()
    return float(labels.mean())

metric_status = pd.DataFrame([
    {"ranking": name, "queue_pages": len(queue),
     "queue_coverage_at_20": len(queue) / K,
     "independent_reviews": int(queue["review_worth_action"].notna().sum()),
     "editorial_precision_at_20": editorial_precision_at_k(queue, K)}
    for name, queue in [("peer-gap proxy", proxy_queue),
                        ("impressions-only baseline", impressions_queue)]
])
display(metric_status)
print("Precision@20 is unavailable: independent editorial reviews have not been supplied.")

,ranking,queue_pages,queue_coverage_at_20,independent_reviews,editorial_precision_at_20
0,peer-gap proxy,20,1.0,0,NaN
1,impressions-only baseline,20,1.0,0,NaN


Precision@20 is unavailable: independent editorial reviews have not been supplied.


## 4. The unit of analysis, as a real dataframe

**One row = one pseudonymized content item (page) in the starter snapshot**, with trailing-90-day counts, not one query, daily event, or client. `content_id` is unique and `client_id` is a grouping key; both remain internal to the analysis, never features. The code displays a small projection of five actual rows, with no titles, URLs, raw queries, or client identifiers. It does not synthesize examples or publish a dataset dump.

The score's unit is the same page. Position-band and content-type summaries below check how much support the lane has. The source export date is not supplied here, so I do not invent calendar boundaries. Engagement rates are also percentages but remain outside this CTR-only score. Missing position is excluded rather than replaced with zero; missing future/editorial labels stay missing.

In [4]:
# Retain source row index so the displayed rows remain identifiable inside the notebook.
page_view = lane[["content_type", "position_band", "avg_position",
                  "impressions_90d", "clicks_90d", "observed_ctr_pct",
                  "peer_ctr_pct", "opportunity_proxy"]].head(5)
assert len(lane) == lane["content_id"].nunique()
assert not {"client_id", "content_id"}.intersection(page_view.columns)
display(page_view.round(4))

grain_summary = grouped.agg(
    pages=("content_id", "size"), clients=("client_id", "nunique"),
    impressions=("impressions_90d", "sum"), clicks=("clicks_90d", "sum")
)
grain_summary["pooled_ctr_pct"] = 100 * grain_summary["clicks"] / grain_summary["impressions"]
assert grain_summary["pages"].sum() == len(lane)
display(grain_summary.round(4))

,content_type,position_band,avg_position,impressions_90d,clicks_90d,observed_ctr_pct,peer_ctr_pct,opportunity_proxy
0,keyword article,>10–20,10.6,3803,29,0.7626,0.3545,0.0000
3,keyword article,>3–10,6.2,11751,58,0.4936,0.4228,0.0000
5,keyword article,>3–10,8.5,3970,1,0.0252,0.3541,13.0563
9,keyword article,>3–10,4.9,1240,2,0.1613,0.4228,3.2422
10,keyword article,1–3,2.2,20919,324,1.5488,0.3514,0.0000


pages  clients  impressions  clicks  \
position_band content_type                                              
1–3           feedly article          3        2         9416     233   
              keyword article       463       15      7426780   36551   
>3–10         comparison article     50        1        80067      34   
              feedly article         83        6       214524     929   
              keyword article      6951       28     88576759  308728   
>10–20        comparison article     11        1        10979      17   
              feedly article         34        3        75191     172   
              keyword article      4414       25     22298826   78355   

                                  pooled_ctr_pct  
position_band content_type                        
1–3           feedly article              2.4745  
              keyword article             0.4922  
>3–10         comparison article          0.0425  
              feedly article              0.4331  
              keyword article             0.3485  
>10–20        comparison article          0.1548  
              feedly article              0.2288  
              keyword article             0.3514

## 5. Why ML beats a fixed rule here

**I have not shown that ML beats a fixed rule yet.** A transparent baseline is the appropriate first step. A single `CTR < 0.5%` threshold ignores exposure, position, content type, and uncertainty. It can flag most eligible pages while giving no useful review order. The calculation below checks that limitation rather than assuming it.

A learned benchmark might eventually handle continuous position, nonlinear exposure effects, intent, freshness, and interactions better than coarse buckets. That only earns its place if it improves independent precision@20 over the simple queue on held-out clients, with stable results and explainable reasons. Average position hides query/device mix; titles and search-result context are absent. More complexity cannot recover those missing causes.

The current proxy also has weaknesses: pooled peers can be dominated by large pages; broad bands and client mix can distort comparisons; a large same-window gap is neither a forecast nor causal evidence. The policy sensitivity table below varies the impression cutoff, but does not establish score stability or model success. Later work should audit peer support, narrower position bands, client sensitivity, uncertainty, and independent top-K reviews before training.

**ML-loop sketch:** define the review decision → check grain/measurement/missingness → build this transparent baseline → obtain independent review judgments → validate by client/cycle → add learning only if it helps → let an editor act → measure later outcomes and monitor changes. Do not use `trend_direction`, `trend_pct`, or the derived decline label as a CTR opportunity target; do not train on IDs, provider/model names, or score-derived labels.

In [5]:
flagged = lane["observed_ctr_pct"].lt(0.5)
print(f"A fixed CTR < 0.5% rule flags {flagged.sum():,}/{len(lane):,} pages ({flagged.mean():.2%}).")
print("This is flag prevalence, not precision or evidence of poor metadata.")

sensitivity = []
for cutoff in [100, 500, 1000]:
    subset = df.loc[df["impressions_90d"].ge(cutoff)
                    & df["avg_position"].between(1, 20)]
    exact_ctr = 100 * subset["clicks_90d"] / subset["impressions_90d"]
    sensitivity.append({"minimum_impressions": cutoff,
                        "eligible_pages": len(subset),
                        "clients": subset["client_id"].nunique(),
                        "pct_flagged_by_fixed_rule": 100 * exact_ctr.lt(0.5).mean()})
display(pd.DataFrame(sensitivity).round(2))
print("No model was trained. No future prediction, edit effect, or model improvement is measured.")

A fixed CTR < 0.5% rule flags 9,773/12,009 pages (81.38%).
This is flag prevalence, not precision or evidence of poor metadata.


,minimum_impressions,eligible_pages,clients,pct_flagged_by_fixed_rule
0,100,15070,29,80.57
1,500,12009,28,81.38
2,1000,9999,28,80.69


No model was trained. No future prediction, edit effect, or model improvement is measured.


**Interpretation of the executed evidence:** the policy yields **12,009 pages across 28 clients**. The peer-support rules cover **11,911 pages** and leave **98** unsupported rather than inventing their benchmark. **8,429** pages have a positive peer gap; that is a proxy count, not the number of actionable edits. Both queues can fill the assumed 20-review budget, but neither has independent review labels.

The fixed 0.5% cutoff flags **9,773 pages (81.38%)**, so the editor still needs an ordering. ML-02 reported 9,745 using the stored two-decimal CTR; this notebook recomputes exact CTR from clicks/impressions, which explains the boundary difference. The threshold sensitivity confirms that volume policy changes the pool size. The peer table also shows that comparison-article groups have only one client, so this notebook appropriately withholds their cross-client benchmarks. These observations support further validation, not a claim that ML already improves CTR.

## Self-check

- [x] Chose ranking/scoring in the CTR / Engagement lane and explained the editor's decision, action, and cost of errors.
- [x] Defined a measured-count-derived opportunity proxy and distinguished it from an independently observed target.
- [x] Named precision@20, a provisional success bar, and a baseline; missing evaluation labels remain explicit.
- [x] Loaded the actual starter slice, verified page grain, and displayed a small real dataframe and the target/proxy column shape.
- [x] Explained what a fixed threshold misses and what evidence would justify a learned model.
- [x] Used percentage units correctly, excluded position 0, checked peer support, and reserved IDs for grouping.
- [x] Ran every code cell top to bottom without errors and saved the outputs.
- [x] Checked outputs for public safety and used measured/observed/decision-support language.

**Remaining evidence gap:** obtain independent editorial judgments before reporting precision@20 or claiming the queue is useful. Neither future CTR nor causal uplift is established here.

**Deliverable:** this executed notebook at `work/notebooks/w02_ml_task_framing.ipynb` inside the starter folder, committed to the GitHub repository.